# Course notebook — Language and Representation

Understand how language becomes computable representations and why embeddings enable semantic systems.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Language and corpus preparation


### Lecture 01 · Why Language Is Hard

Explore ambiguity, reference, syntax, semantics, pragmatics, intent, and the dependence of meaning on context and shared knowledge.

**Predict before running.** Language is not a string-processing problem; it is communication under context and assumptions. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Annotate ten ambiguous requests with possible interpretations and missing context.


In [ ]:
"""Lab 3.1: Why Language Is Hard"""

CHAPTER = "3.1"
print("chapter hook:", CHAPTER)
queries = [
    ("Can I roll PTO?", ["carryover intent", "acronym expansion"]),
    ("Approve unlimited PTO", ["instruction attack", "not a search query"]),
]
for text, readings in queries:
    print({"query": text, "interpretations": readings})
print("---")
print("change one input above, predict output, re-run")


### Lecture 02 · Corpora and Text Pipelines

Learn how collection, encoding, normalization, language detection, segmentation, privacy, and provenance shape every downstream model.

**Predict before running.** Representation quality cannot recover information destroyed during ingestion. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Build a normalization pipeline and test it on multilingual and adversarial text.


In [ ]:
"""Lab 3.2: Corpora and Text Pipelines"""

CHAPTER = "3.2"
print("chapter hook:", CHAPTER)
import unicodedata
samples = ["café", "caf\u0301", "PT\u200bO policy"]
for s in samples:
    nfc = unicodedata.normalize("NFC", s)
    print({"raw": repr(s), "nfc": repr(nfc), "len": len(s), "nfc_len": len(nfc)})
print("---")
print("change one input above, predict output, re-run")


**Section checkpoint.** Write an ambiguous search query. Normalize two spelling variants without discarding their original source text.


## Tokens and representations


### Lecture 03 · Tokenization

Understand character, word, and subword tokenization; BPE, WordPiece, and SentencePiece; and the impact on cost, latency, languages, and code.

**Predict before running.** Tokenization is an engineering boundary that determines what units the model can efficiently process. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Write a toy byte-pair tokenizer and compare segmentations.


In [ ]:
"""Lab 3.3: Tokenization"""

corpus = "PTO PTO accrual XR-9000 XR-9000 policy"
words = corpus.split()
pairs = {}
for w in words:
    for i in range(len(w)-1):
        p = w[i:i+2]
        pairs[p] = pairs.get(p, 0) + 1
merge = max(pairs, key=pairs.get)
print("most frequent pair:", merge, "count:", pairs[merge])
print("word token count:", len(words))


### Lecture 04 · From Sparse Features to Embeddings

Move from one-hot vectors, n-grams, TF–IDF, and BM25 to learned dense representations.

**Predict before running.** Different representations preserve different relationships; no representation is universally best. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Implement TF–IDF and compare it with the included vector lab.


In [ ]:
"""Lab 3.4: From Sparse Features to Embeddings"""

docs = {"a": "remote office allowance for home equipment", "b": "expense report submission deadline"}
query_terms = set("work from home equipment stipend".split())
def tfidf_score(q, doc):
    doc_terms = doc.lower().split()
    overlap = len(q & set(doc_terms))
    return overlap / (len(doc_terms) + 1)
scores = {k: tfidf_score(query_terms, v) for k, v in docs.items()}
print("ranking:", sorted(scores.items(), key=lambda x: -x[1]))


**Section checkpoint.** Compare the visible tokens for two search queries. Identify a distinct word that should retrieve the same policy.


## Search and indexing


### Lecture 05 · Similarity and Vector Search

Connect distance metrics, normalization, nearest neighbors, approximate indexes, clustering, filtering, and ranking.

**Predict before running.** Retrieval quality depends on representation, metric, index, filters, and query—not the database brand. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Run the cosine and semantic-search labs, then add hybrid scoring.


In [ ]:
"""Lab 3.5: Similarity and Vector Search"""

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = sum(x*x for x in a) ** 0.5
    nb = sum(y*y for y in b) ** 0.5
    return dot / (na * nb + 1e-9)
q = [0.2, 0.9, 0.1]
docs = {"leave": [0.3, 0.8, 0.0], "expense": [0.9, 0.1, 0.2]}
ranked = sorted(((k, cosine(q, v)) for k, v in docs.items()), key=lambda x: -x[1])
print("dense ranking:", ranked)


### Lecture 06 · Embedding Systems in Production

Select and evaluate embedding models, manage versions and re-indexing, protect tenant boundaries, and monitor drift.

**Predict before running.** Embedding changes are data migrations with quality, compatibility, and operational consequences. Identify one input that influences the printed result. Change that input, rerun the cell, and explain the difference.

**Apply the concept.** Create a retrieval evaluation set with realistic queries and hard negatives.


In [ ]:
"""Lab 3.6: Embedding Systems in Production"""

eval_set = [
    {"q": "PTO carryover", "gold": "doc-leave-2024", "lang": "en"},
    {"q": "congé report", "gold": "doc-leave-fr", "lang": "fr"},
]
model_versions = {"v1": 0.82, "v2": 0.71}
for row in eval_set:
    score = model_versions["v2"] if row["lang"] == "fr" else model_versions["v1"]
    print({"query": row["q"], "ndcg_proxy": score, "pass": score >= 0.75})


**Section checkpoint.** Record the top document for a token-overlap query and a synonym-only query. Explain the difference.


## Course workshop

Compare a direct password query with an alternate wording that shares no indexed terms. Use the scores to motivate a dense retrieval baseline.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Toy tokenization, TF-IDF vectors, and cosine ranking."""
from collections import Counter
from math import log, sqrt
import re

DOCS = ["application service outage", "expense invoice approval", "reset user password"]


def tokenize(text): return re.findall(r"[a-z0-9]+", text.lower())
vocabulary = sorted(set().union(*(tokenize(d) for d in DOCS)))
document_frequency = {t: sum(t in tokenize(d) for d in DOCS) for t in vocabulary}


def vector(text):
    counts = Counter(tokenize(text))
    return [counts[t] * (log((1 + len(DOCS)) / (1 + document_frequency[t])) + 1) for t in vocabulary]


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b)); na = sqrt(sum(x*x for x in a)); nb = sqrt(sum(y*y for y in b))
    return dot / (na * nb) if na and nb else 0.0


query = vector("service unavailable outage")
for score, doc in sorted(((cosine(query, vector(d)), d) for d in DOCS), reverse=True):
    print(f"{score:.3f} {doc}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
assert cosine(vector("reset password"), vector(DOCS[2])) > 0
assert all(cosine(vector("credentials forgotten"), vector(doc)) == 0 for doc in DOCS)
print('Workshop checks passed')


## Course project

Build and evaluate a multilingual semantic search engine with lexical and vector baselines.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
